# 06_Naive_Bayes

# 06_Scikit_Learn.ipynb

---

# 1. Introduction

Scikit-learn provides several implementations of Naive Bayes, each designed for different types of data.

| Estimator       | Used For                       |
| --------------- | ------------------------------ |
| `GaussianNB`    | Continuous numerical features  |
| `MultinomialNB` | Count data (e.g., word counts) |
| `BernoulliNB`   | Binary features (0/1)          |
| `ComplementNB`  | Imbalanced text classification |
| `CategoricalNB` | Categorical features           |

> **Most common choice:** `GaussianNB` for numerical datasets and `MultinomialNB` for text classification.

In this notebook, we'll primarily focus on **GaussianNB**, while briefly introducing the others.

---

# 2. Import & Constructor

## Required Imports

```python
from sklearn.naive_bayes import GaussianNB

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score
```

---

## Constructor Syntax

```python
GaussianNB(
    *,
    priors=None,
    var_smoothing=1e-9
)
```

---

## Parameter Table

| Parameter       | Default | Description                                                                                  | Common Usage                                       |
| --------------- | ------- | -------------------------------------------------------------------------------------------- | -------------------------------------------------- |
| `priors`        | `None`  | Prior probabilities of classes. If `None`, they are learned from the training data.          | Usually leave as `None`.                           |
| `var_smoothing` | `1e-9`  | Small value added to the variance to avoid division by zero and improve numerical stability. | Increase slightly if numerical instability occurs. |

---

## Recommended Settings

```python
model = GaussianNB()
```

For most datasets, the default settings work well.

---

## Best Practices

* Use **GaussianNB** only when features are continuous.
* Handle missing values before training.
* Although GaussianNB doesn't require feature scaling, scaling can still improve numerical stability on datasets with features of very different magnitudes.
* If a feature has zero variance, adjust `var_smoothing`.

---

# 3. Methods & Attributes

## Methods Table

| Method                 | Purpose                   | Returns     |
| ---------------------- | ------------------------- | ----------- |
| `fit(X, y)`            | Train the model           | `self`      |
| `predict(X)`           | Predict class labels      | NumPy array |
| `predict_proba(X)`     | Probability of each class | NumPy array |
| `predict_log_proba(X)` | Log probabilities         | NumPy array |
| `score(X, y)`          | Mean accuracy             | Float       |
| `get_params()`         | Return model parameters   | Dictionary  |
| `set_params()`         | Update parameters         | `self`      |

---

## Attributes Table

| Attribute        | Description                              |
| ---------------- | ---------------------------------------- |
| `classes_`       | Unique class labels                      |
| `class_count_`   | Number of samples in each class          |
| `class_prior_`   | Prior probability of each class          |
| `theta_`         | Mean of each feature for every class     |
| `var_`           | Variance of each feature for every class |
| `n_features_in_` | Number of input features                 |

---

## Complete Code Example

```python
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Load dataset
iris = load_iris()

X = iris.data
y = iris.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create model
model = GaussianNB()

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))
```

---

## Commonly Used Methods

### `fit()`

```python
model.fit(X_train, y_train)
```

Learns:

* Class priors
* Feature means
* Feature variances

---

### `predict()`

```python
y_pred = model.predict(X_test)
```

Predicts the most likely class.

---

### `predict_proba()`

```python
model.predict_proba(X_test[:5])
```

Example output:

```python
array([
    [0.99, 0.01, 0.00],
    [0.02, 0.97, 0.01],
    ...
])
```

Each row contains the probability of belonging to each class.

---

### `predict_log_proba()`

```python
model.predict_log_proba(X_test[:5])
```

Returns the same probabilities in logarithmic form.

Useful for numerical stability.

---

### `score()`

```python
model.score(X_test, y_test)
```

Returns:

```python
Accuracy
```

Equivalent to:

```python
accuracy_score(y_test, model.predict(X_test))
```

---

## Commonly Used Attributes

### `classes_`

```python
print(model.classes_)
```

Output:

```python
[0 1 2]
```

---

### `class_prior_`

```python
print(model.class_prior_)
```

Example:

```python
[0.33 0.33 0.34]
```

---

### `theta_`

```python
print(model.theta_)
```

Stores the **mean** of every feature for every class.

Shape:

```python
(n_classes, n_features)
```

---

### `var_`

```python
print(model.var_)
```

Stores the **variance** of every feature for every class.

---

# 4. End-to-End Workflow

## Workflow Diagram

```text
Load Data
    ↓
Preprocess
    ↓
Train Model
    ↓
Evaluate
```

---

## Complete Working Example

```python
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Load data
iris = load_iris()

X = iris.data
y = iris.target

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", GaussianNB())
])

# Train
pipeline.fit(X_train, y_train)

# Predict
y_pred = pipeline.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
```

---

## Cross Validation

```python
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    GaussianNB(),
    X,
    y,
    cv=5
)

print(scores)
print("Mean Accuracy:", scores.mean())
```

---

## GridSearchCV

`GaussianNB` has only one major tunable parameter: `var_smoothing`.

```python
from sklearn.model_selection import GridSearchCV
import numpy as np

param_grid = {
    "var_smoothing": np.logspace(0, -9, num=10)
}

grid = GridSearchCV(
    GaussianNB(),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)
```

---

## Model Persistence

Save the trained model:

```python
import joblib

joblib.dump(model, "gaussian_nb.pkl")
```

Load the model:

```python
loaded_model = joblib.load("gaussian_nb.pkl")
```

Predict using the loaded model:

```python
loaded_model.predict(X_test)
```

---

## Important Notes

* GaussianNB assumes each feature follows a Gaussian distribution within each class.
* Scaling is optional but can improve numerical stability.
* `predict_proba()` is useful when you need confidence scores.
* `var_smoothing` is the only commonly tuned hyperparameter.
* Use `MultinomialNB` instead of `GaussianNB` for text data represented as word counts.

---

## Common Errors

| Mistake                             | Why It Happens                                                                 |
| ----------------------------------- | ------------------------------------------------------------------------------ |
| Using `GaussianNB` on word counts   | `MultinomialNB` is more appropriate for count data.                            |
| Not handling missing values         | Naive Bayes in scikit-learn cannot handle `NaN` values directly.               |
| Assuming scaling is mandatory       | GaussianNB works without scaling, though scaling can help numerical stability. |
| Forgetting to split data            | Evaluating on training data gives overly optimistic results.                   |
| Using categorical features directly | Encode categorical variables or use `CategoricalNB`.                           |
| Ignoring `predict_proba()`          | Useful when class probabilities matter, not just the predicted label.          |

---

# Notebook Summary

In this notebook, we covered:

* The different Naive Bayes estimators available in scikit-learn.
* The `GaussianNB` constructor and important parameters.
* Commonly used methods and attributes.
* A complete training and prediction example.
* Building a pipeline.
* Cross-validation.
* Hyperparameter tuning with `GridSearchCV`.
* Saving and loading the trained model.
* Best practices and common mistakes.

---
